# Импорт

In [ ]:
!apt-get install -qq -y quantum-espresso
!pip install -q ase

In [2]:
!apt-get install -qq -y zip > /dev/null 2>&1

In [3]:
import os
import glob
import shutil
import numpy as np
from ase import Atoms
from ase.build import bulk
from ase.io import write, read
from ase.calculators.espresso import Espresso, EspressoProfile

In [4]:
from google.colab import files

# Функции расчёта

In [5]:
def relax(structure, prefix, input_data, pseudopotentials, kpts):
    outdir = f'./tmp_{prefix}'
    os.makedirs(outdir, exist_ok=True)
    calc = Espresso(
        profile=EspressoProfile(command='pw.x', pseudo_dir=os.path.abspath('.')),
        prefix=prefix,
        pseudopotentials=pseudopotentials,
        kpts=kpts,
        input_data=input_data,
        outdir=outdir,
    )
    structure.calc = calc
    energy = structure.get_potential_energy()
    os.replace('espresso.pwi', f'{prefix}/{prefix}.pw.in')
    os.replace('espresso.pwo', f'{prefix}/result/{prefix}.pw.out')
    os.replace('espresso.err', f'{prefix}/result/{prefix}.pw.err')
    print(f'{prefix}: E = {energy:.6f} eV')
    return structure

In [6]:
def run_projwfc(prefix, outdir, degauss=0.015, Emin=-10.0, Emax=20.0, DeltaE=0.01):
    infile = f'{prefix}/{prefix}_projwfc.in'
    outfile = f'{prefix}/result/{prefix}_projwfc.out'
    projwfc_in = f'''
&PROJWFC
  prefix  = '{prefix}'
  outdir  = '{outdir}'
  ngauss  = 1
  degauss = {degauss}
  Emin    = {Emin}
  Emax    = {Emax}
  DeltaE  = {DeltaE}
/
'''
    with open(infile, 'w') as f:
        f.write(projwfc_in)
    os.system(f'projwfc.x < {infile} > {outfile}')
    for f in glob.glob(f'{prefix}.pdos_*'):
        os.replace(f, f'{prefix}/result/{f}')
    if os.path.exists(f'{prefix}.pdos_tot'):
        os.replace(f'{prefix}.pdos_tot', f'{prefix}/result/{prefix}.pdos_tot')
    print(f'{prefix}: projwfc готов')

In [7]:
def run_charge_density(prefix, outdir):
    infile = f'{prefix}/{prefix}_charge.in'
    inputpp = f'''
&INPUTPP
  prefix  = '{prefix}'
  outdir  = '{outdir}'
  filplot = '{prefix}/result/{prefix}.dat'
  plot_num = 0
/
&PLOT
  iflag  = 3
  output_format = 6
  fileout = '{prefix}/result/{prefix}_charge.xsf'
/
'''
    with open(infile, 'w') as f:
        f.write(inputpp)
    os.system(f'pp.x < {infile} > {prefix}/result/{prefix}_charge.out')
    print(f'{prefix}: charge density сохранена')

In [8]:
def run_hartree_potential(prefix, outdir):
    infile = f'{prefix}/{prefix}_hartree.in'
    inputpp = f'''
&INPUTPP
  prefix  = '{prefix}'
  outdir  = '{outdir}'
  filplot = '{prefix}/result/{prefix}.dat'
  plot_num = 11
/
&PLOT
  iflag  = 3
  output_format = 6
  fileout = '{prefix}/result/{prefix}_hartree.xsf'
/
'''
    with open(infile, 'w') as f:
        f.write(inputpp)
    os.system(f'pp.x < {infile} > {prefix}/result/{prefix}_hartree.out')
    print(f'{prefix}: потенциал Хартри сохранён')

# Структуры

In [9]:
Al_start = bulk('Al', 'fcc', a=4.05, cubic=True)
Ni_start = bulk('Ni', 'fcc', a=3.52, cubic=True)

In [10]:
a = 3.57
Ni3Al_start = Atoms('AlNi3', positions=[(0,0,0)]*4, cell=[a,a,a], pbc=True)
Ni3Al_start.set_scaled_positions([(0,0,0), (0.5,0,0.5), (0.5,0.5,0), (0,0.5,0.5)])

# Псевдопотенциалы

In [11]:
if not os.path.exists('Al.pbe-n-kjpaw_psl.1.0.0.UPF'):
    !wget -q http://pseudopotentials.quantum-espresso.org/upf_files/Al.pbe-n-kjpaw_psl.1.0.0.UPF
if not os.path.exists('Ni.pbe-spn-kjpaw_psl.1.0.0.UPF'):
    !wget -q http://pseudopotentials.quantum-espresso.org/upf_files/Ni.pbe-spn-kjpaw_psl.1.0.0.UPF

PBE_ps = {'Ni': 'Ni.pbe-spn-kjpaw_psl.1.0.0.UPF',
          'Al': 'Al.pbe-n-kjpaw_psl.1.0.0.UPF'}

# Параметры расчёта

In [12]:
kpts = (8, 8, 8)
degauss = 0.015

electron_settings = {
    'conv_thr': 1e-8,
    'mixing_mode': 'local-TF',
    'mixing_beta': 0.7,
    'mixing_ndim': 8,
    'diagonalization': 'david',
    'diago_david_ndim': 6,
}

In [13]:
input_data_Al = {
    'control': {'calculation': 'relax', 'restart_mode': 'from_scratch',
                'prefix': 'Al', 'tstress': True, 'tprnfor': True},
    'system': {'ecutwfc': 40, 'ecutrho': 200, 'nspin': 1,
               'occupations': 'smearing', 'smearing': 'methfessel-paxton',
               'degauss': degauss, 'input_dft': 'pbe'},
    'electrons': electron_settings,
    'ions': {'ion_dynamics': 'bfgs'},
    'cell': {'cell_dynamics': 'bfgs', 'press_conv_thr': 0.1},
}

In [14]:
input_data_Ni = {
    'control': {'calculation': 'relax', 'restart_mode': 'from_scratch',
                'prefix': 'Ni', 'tstress': True, 'tprnfor': True},
    'system': {'ecutwfc': 80, 'ecutrho': 500, 'nspin': 2,
               'starting_magnetization(1)': 0.6,
               'occupations': 'smearing', 'smearing': 'methfessel-paxton',
               'degauss': degauss, 'input_dft': 'pbe'},
    'electrons': electron_settings,
    'ions': {'ion_dynamics': 'bfgs'},
    'cell': {'cell_dynamics': 'bfgs', 'press_conv_thr': 0.1},
}

In [15]:
input_data_Ni3Al = {
    'control': {'calculation': 'relax', 'restart_mode': 'from_scratch',
                'prefix': 'Ni3Al', 'tstress': True, 'tprnfor': True},
    'system': {'ecutwfc': 80, 'ecutrho': 500, 'nspin': 2,
               'starting_magnetization(1)': 0.0,
               'starting_magnetization(2)': 0.2,
               'occupations': 'smearing', 'smearing': 'methfessel-paxton',
               'degauss': degauss, 'input_dft': 'pbe'},
    'electrons': electron_settings,
    'ions': {'ion_dynamics': 'bfgs'},
    'cell': {'cell_dynamics': 'bfgs', 'press_conv_thr': 0.1},
}

In [16]:
for prefix in ['Al', 'Ni', 'Ni3Al']:
    os.makedirs(f'{prefix}/result', exist_ok=True)

# Relax

In [18]:
Ni_opt = relax(Ni_start.copy(), 'Ni', input_data_Ni, PBE_ps, kpts)

Ni: E = -23355.944571 eV


In [19]:
Al_opt = relax(Al_start.copy(), 'Al', input_data_Al, PBE_ps, kpts)

Al: E = -2149.865585 eV


In [20]:
Ni3Al_opt = relax(Ni3Al_start.copy(), 'Ni3Al', input_data_Ni3Al, PBE_ps, kpts)

Ni3Al: E = -18056.123865 eV


In [21]:
write('Al/result/Al_opt.cif', Al_opt)
write('Ni/result/Ni_opt.cif', Ni_opt)
write('Ni3Al/result/Ni3Al_opt.cif', Ni3Al_opt)

# DOS (projwfc.x)

In [22]:
run_projwfc('Ni', './tmp_Ni', degauss=degauss)
run_projwfc('Al', './tmp_Al', degauss=degauss)
run_projwfc('Ni3Al', './tmp_Ni3Al', degauss=degauss)

Ni: projwfc готов
Al: projwfc готов
Ni3Al: projwfc готов


# Плотность заряда и потенциал Хартри (pp.x)

In [23]:
run_charge_density('Ni', './tmp_Ni')
run_charge_density('Al', './tmp_Al')
run_charge_density('Ni3Al', './tmp_Ni3Al')
run_hartree_potential('Ni', './tmp_Ni')
run_hartree_potential('Al', './tmp_Al')
run_hartree_potential('Ni3Al', './tmp_Ni3Al')

Ni: charge density сохранена
Al: charge density сохранена
Ni3Al: charge density сохранена
Ni: потенциал Хартри сохранён
Al: потенциал Хартри сохранён
Ni3Al: потенциал Хартри сохранён


In [24]:
for prefix in ['Al', 'Ni', 'Ni3Al']:
    shutil.copy(f'tmp_{prefix}/{prefix}.save/data-file-schema.xml', f'{prefix}/result/data-file-schema.xml')
    shutil.copy(f'tmp_{prefix}/{prefix}.xml', f'{prefix}/result/{prefix}.xml')

# Скачивание результатов

In [25]:
! ls

Al			      Ni3Al			      tmp_Al
Al.pbe-n-kjpaw_psl.1.0.0.UPF  Ni.pbe-spn-kjpaw_psl.1.0.0.UPF  tmp_Ni
Ni			      sample_data		      tmp_Ni3Al


In [26]:
!zip -r QE_results.zip Al/ Ni/ Ni3Al/ 2>/dev/null

files.download('QE_results.zip')

  adding: Al/ (stored 0%)
  adding: Al/result/ (stored 0%)
  adding: Al/result/Al.pdos_atm#2(Al)_wfc#2(p) (deflated 81%)
  adding: Al/result/Al.pdos_atm#1(Al)_wfc#2(p) (deflated 84%)
  adding: Al/result/Al.pw.out (deflated 73%)
  adding: Al/result/Al.pdos_atm#4(Al)_wfc#1(s) (deflated 79%)
  adding: Al/result/Al_charge.xsf (deflated 95%)
  adding: Al/result/Al_opt.cif (deflated 62%)
  adding: Al/result/Al_projwfc.out (deflated 86%)
  adding: Al/result/Al.dat (deflated 87%)
  adding: Al/result/Al_hartree.xsf (deflated 93%)
  adding: Al/result/Al.xml (deflated 85%)
  adding: Al/result/Al.pdos_tot (deflated 78%)
  adding: Al/result/Al.pdos_atm#1(Al)_wfc#1(s) (deflated 79%)
  adding: Al/result/Al.pdos_atm#3(Al)_wfc#2(p) (deflated 82%)
  adding: Al/result/Al_hartree.out (deflated 53%)
  adding: Al/result/Al.pw.err (stored 0%)
  adding: Al/result/Al.pdos_atm#4(Al)_wfc#2(p) (deflated 82%)
  adding: Al/result/Al.pdos_atm#3(Al)_wfc#1(s) (deflated 79%)
  adding: Al/result/Al_charge.out (deflated 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>